In [ ]:
"""
EDUCATIONAL FOOTBALL PREDICTOR - ACCURACY TRACKER v4.0
======================================================
Learn ML model evaluation through football match predictions.
Track accuracy, understand variance, compare to market probabilities.

EDUCATIONAL PURPOSE ONLY - NO BETTING AUTOMATION

Author: Educational Tool
Date: October 2025
License: Educational Use Only
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import warnings
import os
import json
from datetime import datetime
import re

try:
    import tkinter as tk
    from tkinter import ttk, scrolledtext, messagebox
    GUI_AVAILABLE = True
except ImportError:
    GUI_AVAILABLE = False
    print("tkinter not available")

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

CONFIG = {
    'leagues': {
        'E0': 'Premier League',
        'SP1': 'La Liga',
        'D1': 'Bundesliga',
        'I1': 'Serie A',
        'F1': 'Ligue 1',
    },
    'seasons': ['2425', '2324', '2223', '2122', '2021'],
    'base_url': 'https://www.football-data.co.uk/mmz4281',
    'data_folder': 'football_data',
    'log_folder': 'prediction_logs',
    'form_window': 5,
    'random_state': 42,
    'min_logs_required': 5,
}

TEAM_ALIASES = {
    'Man City': 'Manchester City',
    'Man Utd': 'Manchester United',
    'Man United': 'Manchester United',
    'Spurs': 'Tottenham',
    'Tottenham Hotspur': 'Tottenham',
    'Newcastle': 'Newcastle United',
    'West Ham': 'West Ham United',
    'Wolves': 'Wolverhampton Wanderers',
    'Brighton': 'Brighton & Hove Albion',
    'Leicester': 'Leicester City',
}

DISCLAIMER = """
╔══════════════════════════════════════════════════════════════╗
║        EDUCATIONAL TOOL - RESPONSIBLE USE NOTICE             ║
╚══════════════════════════════════════════════════════════════╝

This tool teaches machine learning evaluation and statistics.

KEY EDUCATIONAL FACTS:
• Bookmakers have 5-7% built-in profit margins
• ML models rarely sustain edges over efficient markets
• Need 1000+ outcomes to assess true model accuracy
• Even 60% accurate models lose 40% of the time
• Sports betting is entertainment, not income

LEARNING GOALS:
✓ Practice ML model evaluation
✓ Understand probability vs actual outcomes  
✓ See why beating markets is mathematically hard
✓ Learn about variance and sample sizes

NO betting advice provided. All predictions are theoretical
probabilities for educational comparison only.

Problem gambling help: 1-800-522-4700
══════════════════════════════════════════════════════════════
"""

# ============================================================================
# UTILITIES
# ============================================================================

def ensure_folders():
    for folder in [CONFIG['data_folder'], CONFIG['log_folder']]:
        if not os.path.exists(folder):
            os.makedirs(folder)

def normalize_team(team):
    team = team.strip()
    return TEAM_ALIASES.get(team, team)

# ============================================================================
# LOGGING SYSTEM
# ============================================================================

class PredictionLogger:
    def __init__(self):
        self.log_file = f"{CONFIG['log_folder']}/predictions.json"
        self.logs = self.load_logs()
    
    def load_logs(self):
        if os.path.exists(self.log_file):
            try:
                with open(self.log_file, 'r') as f:
                    return json.load(f)
            except:
                return []
        return []
    
    def save_logs(self):
        with open(self.log_file, 'w') as f:
            json.dump(self.logs, f, indent=2)
    
    def add_log(self, prediction, actual):
        log = {
            'timestamp': datetime.now().isoformat(),
            'match': f"{prediction['home']} vs {prediction['away']}",
            'predicted': self._get_top(prediction['probabilities']),
            'probs': prediction['probabilities'],
            'actual': actual,
            'correct': self._check(prediction, actual)
        }
        self.logs.append(log)
        self.save_logs()
    
    def _get_top(self, probs):
        return max(probs.items(), key=lambda x: x[1])[0]
    
    def _check(self, prediction, actual):
        top = self._get_top(prediction['probabilities'])
        return top == actual.get('result', 'unknown')
    
    def get_stats(self):
        if not self.logs:
            return {
                'total': 0,
                'correct': 0,
                'accuracy': 0,
                'needs_more': CONFIG['min_logs_required']
            }
        
        correct = sum(1 for log in self.logs if log['correct'])
        total = len(self.logs)
        
        return {
            'total': total,
            'correct': correct,
            'accuracy': correct / total if total > 0 else 0,
            'recent_10': self._recent_acc(10),
            'needs_more': max(0, CONFIG['min_logs_required'] - total)
        }
    
    def _recent_acc(self, n):
        recent = self.logs[-n:]
        if not recent:
            return 0
        correct = sum(1 for log in recent if log['correct'])
        return correct / len(recent)
    
    def can_generate(self):
        return len(self.logs) >= CONFIG['min_logs_required']

# ============================================================================
# PARSING
# ============================================================================

def parse_fixtures(text):
    fixtures = []
    lines = text.strip().split('\n')
    
    for line in lines:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        
        if '|' in line and ' vs ' in line:
            parts = [p.strip() for p in line.split('|')]
            if len(parts) >= 1:
                teams = parts[0].split(' vs ')
                if len(teams) == 2:
                    fixtures.append({
                        'HomeTeam': normalize_team(teams[0]),
                        'AwayTeam': normalize_team(teams[1]),
                        'Competition': parts[1] if len(parts) > 1 else 'Unknown',
                        'Kickoff': parts[2] if len(parts) > 2 else ''
                    })
        elif ' vs ' in line:
            teams = line.split(' vs ')
            if len(teams) == 2:
                fixtures.append({
                    'HomeTeam': normalize_team(teams[0]),
                    'AwayTeam': normalize_team(teams[1]),
                    'Competition': 'Unknown',
                    'Kickoff': ''
                })
    
    return fixtures

def parse_odds(text, fixtures):
    odds_map = {}
    lines = text.strip().split('\n')
    
    for line in lines:
        if ':' not in line:
            continue
        
        try:
            match_name, odds_str = line.split(':', 1)
            odds_parts = [float(x.strip()) for x in odds_str.split('|')]
            
            if len(odds_parts) >= 5:
                odds_map[match_name.strip()] = {
                    'home_win': odds_parts[0],
                    'draw': odds_parts[1],
                    'away_win': odds_parts[2],
                    'over_2.5': odds_parts[3],
                    'btts_yes': odds_parts[4]
                }
        except:
            continue
    
    for fixture in fixtures:
        match_key = f"{fixture['HomeTeam']} vs {fixture['AwayTeam']}"
        fixture['odds'] = odds_map.get(match_key, None)
    
    return fixtures

def calc_implied(odds):
    if not odds:
        return None
    
    implied = {
        'home': 1 / odds['home_win'],
        'draw': 1 / odds['draw'],
        'away': 1 / odds['away_win'],
        'over_2.5': 1 / odds['over_2.5'],
        'btts_yes': 1 / odds['btts_yes']
    }
    
    overround = implied['home'] + implied['draw'] + implied['away']
    implied['overround'] = overround
    
    total = implied['home'] + implied['draw'] + implied['away']
    implied['home_norm'] = implied['home'] / total
    implied['draw_norm'] = implied['draw'] / total
    implied['away_norm'] = implied['away'] / total
    
    return implied

# ============================================================================
# DATA LOADING
# ============================================================================

def download_data(leagues, seasons):
    print("=" * 80)
    print("LOADING HISTORICAL DATA")
    print("=" * 80)
    
    all_data = []
    
    for league in leagues:
        print(f"\n{CONFIG['leagues'].get(league, league)}")
        
        for season in seasons:
            filename = f"{CONFIG['data_folder']}/{league}_{season}.csv"
            
            if os.path.exists(filename):
                print(f"  {season}: Cached")
                df = pd.read_csv(filename, encoding='latin-1', encoding_errors='ignore')
            else:
                url = f"{CONFIG['base_url']}/{season}/{league}.csv"
                try:
                    print(f"  {season}: Downloading...")
                    df = pd.read_csv(url, encoding='latin-1', encoding_errors='ignore')
                    df.to_csv(filename, index=False)
                except Exception as e:
                    print(f"  Failed: {e}")
                    continue
            
            df['Season'] = season
            df['League'] = league
            all_data.append(df)
    
    if not all_data:
        raise Exception("No data loaded")
    
    combined = pd.concat(all_data, ignore_index=True)
    print(f"\nTotal: {len(combined)} matches")
    return combined

def update_current_season(leagues):
    print("\nUpdating current season...")
    
    for league in leagues:
        filename = f"{CONFIG['data_folder']}/{league}_2425.csv"
        url = f"{CONFIG['base_url']}/2425/{league}.csv"
        
        try:
            print(f"  {CONFIG['leagues'][league]}...")
            df = pd.read_csv(url, encoding='latin-1', encoding_errors='ignore')
            df.to_csv(filename, index=False)
            print(f"    Updated: {len(df)} matches")
        except Exception as e:
            print(f"    Failed: {e}")

def preprocess_data(df):
    cols = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'Season', 'League']
    available = [c for c in cols if c in df.columns]
    df = df[available].copy()
    
    df = df.dropna(subset=['FTHG', 'FTAG', 'FTR'])
    
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce')
        df = df.dropna(subset=['Date'])
    
    df = df.sort_values('Date').reset_index(drop=True)
    
    df['TotalGoals'] = df['FTHG'] + df['FTAG']
    df['Over2.5'] = (df['TotalGoals'] > 2.5).astype(int)
    df['BTTS'] = ((df['FTHG'] > 0) & (df['FTAG'] > 0)).astype(int)
    
    return df

# ============================================================================
# FEATURES
# ============================================================================

def calc_form(df, team, n=5):
    matches = df[(df['HomeTeam'] == team) | (df['AwayTeam'] == team)].copy()
    
    if len(matches) < n:
        return 5
    
    recent = matches.tail(n)
    points = 0
    
    for _, m in recent.iterrows():
        if m['HomeTeam'] == team:
            points += 3 if m['FTR'] == 'H' else (1 if m['FTR'] == 'D' else 0)
        else:
            points += 3 if m['FTR'] == 'A' else (1 if m['FTR'] == 'D' else 0)
    
    return points

def calc_goals(df, team, n=5, loc='both'):
    if loc == 'home':
        m = df[df['HomeTeam'] == team].tail(n)
        scored = m['FTHG'].mean() if len(m) > 0 else 1.5
        conceded = m['FTAG'].mean() if len(m) > 0 else 1.0
    elif loc == 'away':
        m = df[df['AwayTeam'] == team].tail(n)
        scored = m['FTAG'].mean() if len(m) > 0 else 1.2
        conceded = m['FTHG'].mean() if len(m) > 0 else 1.3
    else:
        home = df[df['HomeTeam'] == team].tail(n)
        away = df[df['AwayTeam'] == team].tail(n)
        total = len(home) + len(away)
        scored = (home['FTHG'].sum() + away['FTAG'].sum()) / max(total, 1)
        conceded = (home['FTAG'].sum() + away['FTHG'].sum()) / max(total, 1)
    
    return scored, conceded

def engineer_features(df, window=5):
    features = []
    
    print(f"\nEngineering features...")
    
    for idx in range(window, len(df)):
        match = df.iloc[idx]
        history = df.iloc[:idx]
        
        home = match['HomeTeam']
        away = match['AwayTeam']
        
        h_form = calc_form(history, home, window)
        a_form = calc_form(history, away, window)
        
        h_gs, h_gc = calc_goals(history, home, window, 'home')
        a_gs, a_gc = calc_goals(history, away, window, 'away')
        
        features.append({
            'HomeTeam': home,
            'AwayTeam': away,
            'home_form': h_form,
            'away_form': a_form,
            'home_goals_scored': h_gs,
            'away_goals_scored': a_gs,
            'home_goals_conceded': h_gc,
            'away_goals_conceded': a_gc,
            'form_diff': h_form - a_form,
            'FTR': match['FTR'],
            'Over2.5': match['Over2.5'],
            'BTTS': match['BTTS']
        })
    
    print(f"Created {len(features)} samples")
    return pd.DataFrame(features)

# ============================================================================
# TRAINING
# ============================================================================

def train_models(features_df):
    print("\n" + "=" * 80)
    print("TRAINING MODELS")
    print("=" * 80)
    
    feat_cols = [
        'home_form', 'away_form',
        'home_goals_scored', 'away_goals_scored',
        'home_goals_conceded', 'away_goals_conceded',
        'form_diff'
    ]
    
    X = features_df[feat_cols]
    
    split_idx = int(len(X) * 0.8)
    X_train = X.iloc[:split_idx]
    X_test = X.iloc[split_idx:]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    models = {}
    
    print("\n1. Match Result (1X2)")
    y_train = features_df.iloc[:split_idx]['FTR']
    y_test = features_df.iloc[split_idx:]['FTR']
    
    m = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
    m.fit(X_train_scaled, y_train)
    
    acc = accuracy_score(y_test, m.predict(X_test_scaled))
    print(f"   Accuracy: {acc:.2%}")
    models['result'] = m
    
    print("\n2. Over/Under 2.5")
    y_train = features_df.iloc[:split_idx]['Over2.5']
    y_test = features_df.iloc[split_idx:]['Over2.5']
    
    m = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)
    m.fit(X_train_scaled, y_train)
    
    acc = accuracy_score(y_test, m.predict(X_test_scaled))
    print(f"   Accuracy: {acc:.2%}")
    models['over'] = m
    
    print("\n3. BTTS")
    y_train = features_df.iloc[:split_idx]['BTTS']
    y_test = features_df.iloc[split_idx:]['BTTS']
    
    m = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)
    m.fit(X_train_scaled, y_train)
    
    acc = accuracy_score(y_test, m.predict(X_test_scaled))
    print(f"   Accuracy: {acc:.2%}")
    models['btts'] = m
    
    print("\n" + "=" * 80)
    
    return models, scaler, feat_cols

# ============================================================================
# PREDICTION
# ============================================================================

def predict_match(models, scaler, feat_cols, fixture, historical_df):
    home = fixture['HomeTeam']
    away = fixture['AwayTeam']
    
    h_form = calc_form(historical_df, home, CONFIG['form_window'])
    a_form = calc_form(historical_df, away, CONFIG['form_window'])
    
    h_gs, h_gc = calc_goals(historical_df, home, CONFIG['form_window'], 'home')
    a_gs, a_gc = calc_goals(historical_df, away, CONFIG['form_window'], 'away')
    
    feats = {
        'home_form': h_form,
        'away_form': a_form,
        'home_goals_scored': h_gs,
        'away_goals_scored': a_gs,
        'home_goals_conceded': h_gc,
        'away_goals_conceded': a_gc,
        'form_diff': h_form - a_form
    }
    
    X = pd.DataFrame([feats])[feat_cols]
    X_scaled = scaler.transform(X)
    
    result_proba = models['result'].predict_proba(X_scaled)[0]
    classes = models['result'].classes_
    probs = dict(zip(classes, result_proba))
    
    h_prob = probs.get('H', 0.33)
    d_prob = probs.get('D', 0.33)
    a_prob = probs.get('A', 0.33)
    
    over_prob = models['over'].predict_proba(X_scaled)[0][1]
    btts_prob = models['btts'].predict_proba(X_scaled)[0][1]
    
    implied = None
    if fixture.get('odds'):
        implied = calc_implied(fixture['odds'])
    
    return {
        'home': home,
        'away': away,
        'competition': fixture.get('Competition', 'Unknown'),
        'kickoff': fixture.get('Kickoff', ''),
        'probabilities': {
            'home': h_prob,
            'draw': d_prob,
            'away': a_prob,
            'over_2.5': over_prob,
            'btts_yes': btts_prob
        },
        'implied_probs': implied
    }

# ============================================================================
# GUI
# ============================================================================

class EducationalPredictorGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Educational Football Predictor - Accuracy Tracker v4.0")
        self.root.geometry("1400x900")
        
        self.logger = PredictionLogger()
        self.models = None
        self.predictions = []
        
        self.show_disclaimer()
        self.create_widgets()
    
    def show_disclaimer(self):
        popup = tk.Toplevel(self.root)
        popup.title("Educational Tool")
        popup.geometry("700x450")
        
        text = scrolledtext.ScrolledText(popup, wrap=tk.WORD, font=('Courier', 9))
        text.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        text.insert('1.0', DISCLAIMER)
        text.config(state=tk.DISABLED)
        
        ttk.Button(popup, text="I Understand", 
                  command=popup.destroy).pack(pady=10)
    
    def create_widgets(self):
        main = ttk.Frame(self.root, padding="10")
        main.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Title
        title_frame = ttk.Frame(main)
        title_frame.grid(row=0, column=0, columnspan=2, pady=10)
        
        ttk.Label(title_frame, text="Educational Football Predictor", 
                 font=('Arial', 18, 'bold')).pack()
        ttk.Label(title_frame, text="Accuracy Tracker v4.0", 
                 font=('Arial', 10, 'italic')).pack()
        
        # Controls
        ctrl = ttk.LabelFrame(main, text="Controls", padding="10")
        ctrl.grid(row=1, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=10)
        
        ttk.Label(ctrl, text="Leagues:").grid(row=0, column=0, padx=5)
        self.league_var = tk.StringVar(value="Premier League")
        ttk.Combobox(ctrl, textvariable=self.league_var, 
                    values=["Premier League", "Top 5 European"], 
                    width=18).grid(row=0, column=1, padx=5)
        
        ttk.Button(ctrl, text="Update Data", 
                  command=self.update_data).grid(row=0, column=2, padx=5)
        ttk.Button(ctrl, text="Load & Train", 
                  command=self.train).grid(row=0, column=3, padx=5)
        ttk.Button(ctrl, text="Log Outcomes", 
                  command=self.log_outcomes).grid(row=0, column=4, padx=5)
        
        # Input
        input_frame = ttk.Frame(main)
        input_frame.grid(row=2, column=0, sticky=(tk.W, tk.E, tk.N, tk.S), padx=(0, 5))
        
        ttk.Label(input_frame, text="Fixtures", 
                 font=('Arial', 10, 'bold')).pack(anchor=tk.W)
        
        self.fixture_text = scrolledtext.ScrolledText(input_frame, height=10, width=60, 
                                                     font=('Consolas', 9))
        self.fixture_text.pack(fill=tk.BOTH, expand=True, pady=5)
        
        self.fixture_text.insert('1.0', "Arsenal vs West Ham | Premier League | 17:00")
        
        ttk.Button(input_frame, text="Parse Fixtures", 
                  command=self.parse_fixtures).pack(fill=tk.X, pady=5)
        
        ttk.Label(input_frame, text="Odds (Optional)", 
                 font=('Arial', 10, 'bold')).pack(anchor=tk.W, pady=(10, 0))
        
        self.odds_text = scrolledtext.ScrolledText(input_frame, height=8, width=60, 
                                                  font=('Consolas', 9))
        self.odds_text.pack(fill=tk.BOTH, expand=True, pady=5)
        
        self.odds_text.insert('1.0', "Arsenal vs West Ham: 1.45|4.50|6.00|1.80|1.70")
        
        ttk.Button(input_frame, text="Parse Odds", 
                  command=self.parse_odds).pack(fill=tk.X, pady=5)
        
        self.predict_btn = ttk.Button(input_frame, text="Generate Predictions", 
                                     command=self.predict, state=tk.DISABLED)
        self.predict_btn.pack(fill=tk.X, pady=10)
        
        # Tabs
        notebook = ttk.Notebook(main)
        notebook.grid(row=2, column=1, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Predictions
        pred_frame = ttk.Frame(notebook)
        notebook.add(pred_frame, text="Predictions")
        
        self.tree = ttk.Treeview(pred_frame, 
                                columns=('Home', 'Away', 'H%', 'D%', 'A%', 'O2.5%', 'BTTS%', 'Mkt%'),
                                show='headings', height=20)
        
        for col, width in [('Home', 120), ('Away', 120), ('H%', 60), ('D%', 60), 
                          ('A%', 60), ('O2.5%', 60), ('BTTS%', 60), ('Mkt%', 60)]:
            self.tree.heading(col, text=col)
            self.tree.column(col, width=width)
        
        scroll = ttk.Scrollbar(pred_frame, orient=tk.VERTICAL, command=self.tree.yview)
        self.tree.configure(yscroll=scroll.set)
        
        self.tree.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        scroll.grid(row=0, column=1, sticky=(tk.N, tk.S))
        
        # Dashboard
        dash_frame = ttk.Frame(notebook)
        notebook.add(dash_frame, text="Dashboard")
        
        self.dash_text = scrolledtext.ScrolledText(dash_frame, height=25, width=80, 
                                                  font=('Consolas', 10))
        self.dash_text.pack(fill=tk.BOTH, expand=True)
        
        self.update_dashboard()
        
        # Log history
        log_frame = ttk.Frame(notebook)
        notebook.add(log_frame, text="History")
        
        self.log_text = scrolledtext.ScrolledText(log_frame, height=25, width=80, 
                                                 font=('Consolas', 9))
        self.log_text.pack(fill=tk.BOTH, expand=True)
        
        self.update_log_history()
        
        # Status
        self.status = tk.StringVar(value="Ready")
        ttk.Label(main, textvariable=self.status, relief=tk.SUNKEN, 
                 anchor=tk.W).grid(row=3, column=0, columnspan=2, sticky=(tk.W, tk.E))
        
        # Weights
        self.root.columnconfigure(0, weight=1)
        self.root.rowconfigure(0, weight=1)
        main.columnconfigure(1, weight=2)
        main.columnconfigure(0, weight=1)
        main.rowconfigure(2, weight=1)
    
    def update_data(self):
        self.status.set("Updating...")
        self.root.update()
        
        try:
            leagues = ['E0'] if self.league_var.get() == "Premier League" else ['E0', 'SP1', 'D1', 'I1', 'F1']
            update_current_season(leagues)
            self.status.set("Updated")
            messagebox.showinfo("Success", "Data updated")
        except Exception as e:
            messagebox.showerror("Error", str(e))
    
    def train(self):
        self.status.set("Loading...")
        self.root.update()
        
        try:
            leagues = ['E0'] if self.league_var.get() == "Premier League" else ['E0', 'SP1', 'D1', 'I1', 'F1']
            
            df = download_data(leagues, CONFIG['seasons'])
            df = preprocess_data(df)
            self.historical_df = df
            
            self.status.set("Training...")
            self.root.update()
            
            features = engineer_features(df)
            self.models, self.scaler, self.feat_cols = train_models(features)
            
            self.status.set("Ready")
            self.predict_btn.config(state=tk.NORMAL)
            messagebox.showinfo("Success", "Models trained")
        except Exception as e:
            messagebox.showerror("Error", str(e))
    
    def parse_fixtures(self):
        text = self.fixture_text.get('1.0', tk.END)
        fixtures = parse_fixtures(text)
        
        if fixtures:
            self.fixtures = fixtures
            self.status.set(f"Parsed {len(fixtures)} fixtures")
        else:
            messagebox.showwarning("Warning", "No fixtures found")
    
    def parse_odds(self):
        if not hasattr(self, 'fixtures'):
            messagebox.showwarning("Warning", "Parse fixtures first")
            return
        
        text = self.odds_text.get('1.0', tk.END)
        self.fixtures = parse_odds(text, self.fixtures)
        
        with_odds = sum(1 for f in self.fixtures if f.get('odds'))
        self.status.set(f"Odds parsed: {with_odds}/{len(self.fixtures)}")
    
    def predict(self):
        if not self.models:
            messagebox.showwarning("Warning", "Train first")
            return
        
        if not hasattr(self, 'fixtures'):
            messagebox.showwarning("Warning", "Parse fixtures first")
            return
        
        stats = self.logger.get_stats()
        if stats['needs_more'] > 0:
            response = messagebox.askyesno(
                "Logging Required",
                f"Log {stats['needs_more']} more outcomes first?\n\n"
                "This helps track accuracy. Continue anyway?"
            )
            if not response:
                return
        
        self.status.set("Predicting...")
        self.root.update()
        
        try:
            for item in self.tree.get_children():
                self.tree.delete(item)
            
            self.predictions = []
            
            for fixture in self.fixtures:
                pred = predict_match(self.models, self.scaler, self.feat_cols, 
                                   fixture, self.historical_df)
                self.predictions.append(pred)
                
                p = pred['probabilities']
                impl = pred['implied_probs']
                
                mkt = f"{impl['home_norm']:.1%}" if impl else "N/A"
                
                self.tree.insert('', 'end', values=(
                    pred['home'], pred['away'],
                    f"{p['home']:.1%}", f"{p['draw']:.1%}", f"{p['away']:.1%}",
                    f"{p['over_2.5']:.1%}", f"{p['btts_yes']:.1%}",
                    mkt
                ))
            
            self.status.set(f"Predicted {len(self.predictions)} matches")
        except Exception as e:
            messagebox.showerror("Error", str(e))
    
    def log_outcomes(self):
        if not self.predictions:
            messagebox.showinfo("Info", "Generate predictions first")
            return
        
        dialog = tk.Toplevel(self.root)
        dialog.title("Log Outcomes")
        dialog.geometry("500x400")
        
        ttk.Label(dialog, text="Log actual results", 
                 font=('Arial', 11, 'bold')).pack(pady=10)
        
        for pred in self.predictions[:5]:
            frame = ttk.LabelFrame(dialog, text=f"{pred['home']} vs {pred['away']}", padding="5")
            frame.pack(fill=tk.X, padx=10, pady=5)
            
            result_var = tk.StringVar(value="H")
            ttk.Radiobutton(frame, text="Home", variable=result_var, value="H").pack(side=tk.LEFT)
            ttk.Radiobutton(frame, text="Draw", variable=result_var, value="D").pack(side=tk.LEFT)
            ttk.Radiobutton(frame, text="Away", variable=result_var, value="A").pack(side=tk.LEFT)
            
            ttk.Button(frame, text="Log", 
                      command=lambda p=pred, r=result_var: self.save_log(p, r.get(), dialog)).pack(side=tk.RIGHT)
    
    def save_log(self, pred, result, dialog):
        self.logger.add_log(pred, {'result': result})
        self.update_dashboard()
        self.update_log_history()
        self.status.set(f"Logged. Total: {len(self.logger.logs)}")
        messagebox.showinfo("Logged", "Outcome recorded")
        dialog.destroy()
    
    def update_dashboard(self):
        self.dash_text.delete('1.0', tk.END)
        
        stats = self.logger.get_stats()
        
        self.dash_text.insert(tk.END, "ACCURACY DASHBOARD\n")
        self.dash_text.insert(tk.END, "=" * 70 + "\n\n")
        
        if stats['total'] == 0:
            self.dash_text.insert(tk.END, "No logs yet. Start predicting and logging.\n")
        else:
            self.dash_text.insert(tk.END, f"Total Logged: {stats['total']}\n")
            self.dash_text.insert(tk.END, f"Correct: {stats['correct']}\n")
            self.dash_text.insert(tk.END, f"Accuracy: {stats['accuracy']:.2%}\n\n")
            
            if stats['total'] < 30:
                self.dash_text.insert(tk.END, f"Sample too small. Need {30 - stats['total']} more.\n")
            elif stats['total'] < 100:
                self.dash_text.insert(tk.END, f"Decent sample. Target: 100+\n")
            else:
                self.dash_text.insert(tk.END, f"Good sample size\n")
    
    def update_log_history(self):
        self.log_text.delete('1.0', tk.END)
        
        if not self.logger.logs:
            self.log_text.insert(tk.END, "No logs yet\n")
            return
        
        self.log_text.insert(tk.END, "LOG HISTORY\n")
        self.log_text.insert(tk.END, "=" * 70 + "\n\n")
        
        for i, log in enumerate(reversed(self.logger.logs[-20:]), 1):
            date = datetime.fromisoformat(log['timestamp']).strftime('%Y-%m-%d')
            correct = "✓" if log['correct'] else "✗"
            
            self.log_text.insert(tk.END, f"{i}. {date} - {log['match']}\n")
            self.log_text.insert(tk.END, f"   Pred: {log['predicted']} | Act: {log['actual']} {correct}\n\n")

def main():
    ensure_folders()
    
    if GUI_AVAILABLE:
        root = tk.Tk()
        app = EducationalPredictorGUI(root)
        root.mainloop()
    else:
        print("GUI not available")
        print(DISCLAIMER)

if __name__ == "__main__":
    main()


████████████████████████████████████████████████████████████████████████████████
                █  FOOTBALL PREDICTION SYSTEM - IMPROVED VERSION                █
████████████████████████████████████████████████████████████████████████████████

Which leagues would you like to include?
1. Premier League only (fastest)
2. Premier League + Champions League (recommended)
3. Top 5 European Leagues
4. All available leagues



Select option (1-4, default 2):  2


STEP 1: DOWNLOADING HISTORICAL DATA

📁 Premier League (E0)
--------------------------------------------------------------------------------
  ✓ 2425: Loading from local file
    Loaded 380 matches
  ✓ 2324: Loading from local file
    Loaded 380 matches
  ✓ 2223: Loading from local file
    Loaded 380 matches
  ✓ 2122: Loading from local file
    Loaded 380 matches
  ✓ 2021: Loading from local file
    Loaded 380 matches

📁 Champions League (EC)
--------------------------------------------------------------------------------
  ✓ 2425: Loading from local file
    Loaded 552 matches
  ✓ 2324: Loading from local file
    Loaded 552 matches
  ✓ 2223: Loading from local file
    Loaded 552 matches
  ✓ 2122: Loading from local file
    Loaded 506 matches
  ✓ 2021: Loading from local file
    Loaded 477 matches

✓ Total matches loaded: 4539 across 2 leagues

STEP 2: PREPROCESSING DATA
✓ Removed 0 rows with missing data
✓ Cleaned dataset: 4539 matches
✓ Date range: 2020-09-12 00:00:00 to 2025-


Select option (1-3, default 1):  1



📝 MANUAL FIXTURE ENTRY
Enter matches in format: HomeTeam vs AwayTeam
Examples:
  - Arsenal vs Man City
  - Real Madrid vs Barcelona
  - Bayern Munich vs Dortmund

Type 'done' when finished, or press Enter to skip



Match 1:  Atalanta vs. Club Brugge


  ⚠️  Invalid format. Use: HomeTeam vs AwayTeam


Match 1:  Atalanta vs Club Brugge
Match 2:  Kairat Almaty vs Real Madrid
Match 3:  Atlético Madrid vs Eintracht Frankfurt
Match 4:  Chelsea vs Benfica
Match 5:  Galatasaray vs Liverpool
Match 6:  Inter Milan vs Slavia Prague
Match 7:  Marseille vs Ajax
Match 8:  Pafos vs Bayern Munich
Match 9:  Bodø/Glimt vs Tottenham
Match 10:  done



✓ Added 9 matches

STEP 5: PREDICTING TODAY'S MATCHES
Date: Tuesday, September 30, 2025

🏟️  Atalanta vs Club Brugge
    Competition: Manual Entry
--------------------------------------------------------------------------------
⚠️  WARNING: Limited historical data for these teams
   Predictions may be less reliable
📊 Match Result:
   Atalanta Win: 47.0%
   Draw:           8.0%
   Club Brugge Win: 45.0%

⚽ Goals:
   Over 2.5:  76.2%
   Under 2.5: 23.8%

🎯 BTTS:
   Yes: 64.1%
   No:  35.9%

🏟️  Kairat Almaty vs Real Madrid
    Competition: Manual Entry
--------------------------------------------------------------------------------
⚠️  WARNING: Limited historical data for these teams
   Predictions may be less reliable
📊 Match Result:
   Kairat Almaty Win: 47.0%
   Draw:           8.0%
   Real Madrid Win: 45.0%

⚽ Goals:
   Over 2.5:  76.2%
   Under 2.5: 23.8%

🎯 BTTS:
   Yes: 64.1%
   No:  35.9%

🏟️  Atlético Madrid vs Eintracht Frankfurt
    Competition: Manual Entry
-----------------